In [1]:
import os
from datetime import datetime
import openpyxl
from sqlalchemy import create_engine
from sqlalchemy_utils import database_exists, create_database
from datetime import date
from sqlalchemy import ForeignKey, String, Date, Numeric, Integer, CheckConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker
from sqlalchemy import inspect


# 1. Creación de Base de Datos (SQLite) con SQLAlchemy  

Crearemos una base de datos **SQLite** en el `database_path` indicado.  
Si ya existe una base de datos en ese directorio, será eliminada y creada de nuevo.

In [2]:
database_path = "../Database/Petrola.db"
database_url = f"sqlite:///{database_path}"


engine = create_engine(database_url)
if not database_exists(engine.url):
    create_database(engine.url)
    print(f"Database successfully created at {database_path}")
else:
    print(f"Database already exists at {database_path}")

Database already exists at ../Database/Petrola.db


# 2. Creación de las Tablas

Con el apartado de análisis de datos, hemos llegado a las siguientes conclusiones:  

- No hay filas duplicadas, no tenemos que preocuparnos a la hora de la insercción.  
- **`Library RT`** y **`Group`** deben admitir valores perdidos (Nulos) 
- **`Match Factor`** representa un porcentaje y debe tratarse como tal.  

Para crear las tablas usamos **SQLAlchemy ORM**, de esta forma es más visual y sencillo.  
Hemos decidido dividir los atributos en tres tablas diferentes:  

### **Tabla Compounds**  
- `cas` (Primary Key)  
- `name`  
- `formula`  
- `group` (nullable)  

### **Tabla Stations**  
- `station_id` (Primary Key)
- `st_type`
- `geology`
- `x` 
- `y`

### **Tabla Samples**  
- `id` (Primary Key generada automáticamente)  
- `station_id` (Foreign Key a `Stations.station_id`)  
- `compound_cas` (Foreign Key a `Compounds.cas`)  
- `component_rt`  
- `library_rt` (nullable)  
- `match_factor` (con restricción entre 0 y 100)  
- `sample_date` (tipo Date)  

Consultar la imagen `Entity Relationship Diagram.png` de la carpeta `/Database` para visualizar la estructura de las tablas.

In [3]:
Session = sessionmaker(bind=engine)
session = Session()

class Base(DeclarativeBase):
    pass

class Stations(Base):
    __tablename__ = "stations"
    
    station_id: Mapped[str] = mapped_column(String, primary_key=True)
    st_type: Mapped[str] = mapped_column(String(20), nullable=True)
    geology: Mapped[str] = mapped_column(String(20), nullable=True)
    x: Mapped[float] = mapped_column(Numeric(10,3), nullable=False)
    y: Mapped[float] = mapped_column(Numeric(10,3), nullable=False)
    
class Compounds(Base):
    __tablename__ = "compounds"
    
    cas: Mapped[str] = mapped_column(String(20), primary_key=True)
    name: Mapped[str] = mapped_column(String(80), nullable=False)
    formula: Mapped[str] = mapped_column(String(20), nullable=False)
    group: Mapped[str] = mapped_column(String(30), nullable=True)

class Samples(Base):
    __tablename__ = "samples"
    
    id: Mapped[int] = mapped_column(Integer, autoincrement=True, primary_key=True)
    station_id: Mapped[str] = mapped_column(ForeignKey("stations.station_id"), nullable=False)
    compound_cas: Mapped[str] = mapped_column(ForeignKey("compounds.cas"), nullable=False)
    component_rt: Mapped[float] = mapped_column(Numeric(17,13), nullable=False) #13 digitos en la parte decimal -> (0000.0000000000000 - 9999.9999999999999)
    library_rt: Mapped[float] = mapped_column(Numeric(17,13), nullable=True)
    match_factor: Mapped[float] = mapped_column(Numeric(16, 13), nullable=False) # como es un percentaje la parte entera admite solo 3 digitos y la decimal 13 digitos
    sample_date: Mapped[date] = mapped_column(Date, nullable=False)

    __table_args__ = (
        # No podemos insertar un porcentaje negativo o mayor a 100%
        CheckConstraint("match_factor >= 0 AND match_factor <= 100", name="check_match_factor_range"),
    )

Base.metadata.create_all(engine)
print("Database Tables created")

Database Tables created


In [4]:
# Consultamos para saber si las tablas se han creado correctamente
inspector = inspect(engine)
tables = inspector.get_table_names()
for table in tables:
    print(f"Tabla: {table}")
    columns = inspector.get_columns(table)
    for column in columns:
        print(f"    Columna: {column['name']} - Tipo: {column['type']}")

Tabla: compounds
    Columna: cas - Tipo: VARCHAR(20)
    Columna: name - Tipo: VARCHAR(80)
    Columna: formula - Tipo: VARCHAR(20)
    Columna: group - Tipo: VARCHAR(30)
Tabla: samples
    Columna: id - Tipo: INTEGER
    Columna: station_id - Tipo: VARCHAR
    Columna: compound_cas - Tipo: VARCHAR(20)
    Columna: component_rt - Tipo: NUMERIC(17, 13)
    Columna: library_rt - Tipo: NUMERIC(17, 13)
    Columna: match_factor - Tipo: NUMERIC(16, 13)
    Columna: sample_date - Tipo: DATE
Tabla: stations
    Columna: station_id - Tipo: VARCHAR
    Columna: st_type - Tipo: VARCHAR(20)
    Columna: geology - Tipo: VARCHAR(20)
    Columna: well_depth - Tipo: NUMERIC(8, 3)
    Columna: x - Tipo: NUMERIC(10, 3)
    Columna: y - Tipo: NUMERIC(10, 3)


# 3. Inserción de Tuplas  (Stations)

Primero tenemos que insertar las tuplas de la tabla Stations, por lo que recorremos el excel de la estaciones y las rellenamos una una.  

In [5]:
# Diccionario de conversión de la variable TIPO
tipo_dic = {
    0: "DESCONOCIDO",
    1: "SONDEO",
    2: "LAGUNA",
    3: "SURGENTE",
    4: "MANANTIAL",
    5: "DEPURADORA",
    6: "POZO EXCAVADO",
    7: "BALSA DE RIEGO",
    8: "ESTACIÓN METEOROLÓGICA (EMP)"
}


# Ruta del Excel
excel_path_input = "../Datos Excel/Estaciones/General_completo.xlsx"

# Cargamos el primer excel
workbook = openpyxl.load_workbook(excel_path_input)
station_tuples = 0
i = 0
total_rows = sum(sheet.max_row - 1 for sheet in workbook.worksheets)

# Creamos las columnas Date y Group
for sheet in workbook.worksheets:
    for row in sheet.iter_rows(min_row=2):
        i+=1
        #print(f"Row {i}/{total_rows}")
        print(f"\rRow {i}/{total_rows} - {((i/total_rows)*100):.2f}%", end="")  # Sobrescribe la línea con el nuevo progreso
        # row[0] = station_id | row[1] = st_type	| row[2] = geology	| row[3] = well_depth	
        # row[4] = elevation	    | row[5] = x	        | row[6] = y
        station = Stations(
            station_id=row[0].value,
            st_type=tipo_dic[int(row[1].value)],
            geology=row[2].value if row[2].value is not None and row[2].value != "" else 'DESCONOCIDO',
            x=row[5].value,
            y=row[6].value
        )
        # Verificar si la estacion ya ha sido añadida a la base de datos
        existing_station = session.query(Stations).filter_by(station_id=row[0].value).first()
        if not existing_station:
            session.add(station)
            station_tuples+=1
            
# Guardar los cambios en la base de datos (ESTO ESTÁ CAMBIADO)
session.commit()
print("\nAll tuples were successfully inserted into the database.")
print(f"{station_tuples} tuples inserted into Stations")

Row 109/109 - 100.00%
All tuples were successfully inserted into the database.
109 tuples inserted into Stations


In [6]:
# Comprobamos que la stations se han insertado bien
engine = create_engine("sqlite:///../Database/Petrola.db")

Session = sessionmaker(bind=engine)
session = Session()

all_stations = session.query(Stations).all()

for st in all_stations:
    print(f"{st.station_id} - {st.st_type} - {st.geology} - {st.x} - {st.y}")

493 - SONDEO - JURÁSICO - 628525.000 - 4302550.000
699 - SONDEO - JURÁSICO - 619500.000 - 4300650.000
2429 - SONDEO - JURÁSICO - 631340.000 - 4300490.000
2430 - SONDEO - JURÁSICO - 619644.000 - 4300550.000
2534 - SONDEO - CRETÁCICO - 622443.000 - 4300541.000
2535 - SONDEO - CRETÁCICO - 626300.000 - 4301891.000
2536 - MANANTIAL - CRETÁCICO - 626734.000 - 4301563.000
2537 - SONDEO - CRETÁCICO - 625472.000 - 4301901.000
2538 - SONDEO - CRETÁCICO - 621632.000 - 4299836.000
2539 - SONDEO - CRETÁCICO - 621437.000 - 4299801.000
2540 - SONDEO - CRETÁCICO - 624879.000 - 4301173.000
2541 - SONDEO - CRETÁCICO - 620857.000 - 4302948.000
2542 - SONDEO - CRETÁCICO - 620806.000 - 4302927.000
2543 - SONDEO - JURÁSICO - 624630.000 - 4298639.000
2544 - SONDEO - CRETÁCICO - 625135.000 - 4299722.000
2545 - SONDEO - CRETÁCICO - 624284.000 - 4299623.000
2546 - SONDEO - CRETÁCICO - 623699.000 - 4297864.000
2547 - SONDEO - CRETÁCICO - 623258.000 - 4297875.000
2548 - SURGENTE - CRETÁCICO - 627659.000 - 4299652

# 4. Insercción de tuplas (Samples y Compound)

In [7]:
# Funciones auxiliares para la insercción
def sheet_is_valid(sheet):
    return not len(sheet.title) < 8

def row_is_valid(row):
    for i in range(7):
        if row[i].value not in (None, "", ''):
            return True
    return False

def excel_is_valid(excel):
    return not excel[0:2] == "GW"

# Función para poner todas las fechas en el mismo formato: 'YYYY-MM-DD'
def normalize_dates(date_list):
    output_list = []
    dt = None
    year = None
    month = None
    for date in date_list:
        try:
            # Algunas de las fechas tienen espacios, los eliminamos
            date = date.replace(" ", "")
            
            # El año siempre viene defnido por los 4 ultimos caracteres y el mes por los 2 anteriores al año
            # Algunos de los dias vienen no definidos excatemente (ej: 17_19022020), nos quedamos con el último dia del rango por comodidad
            year = date[-4:]
            month = date[-6:-4]
            day = date[-8:-6]
            
            # Creamos un datetime y lo formateamos a YYYY-MM-DD (nos vendrá bien este formato para insertar en la base de datos)
            dt = datetime(int(year), int(month), int(day))
            output_list.append(dt.strftime("%Y-%m-%d"))
        except Exception as e:
            print(f"\nError: {e}")
            print(f"Nombre de página incorrecto: {date_list[0]}, por favor corrija el nombre al formato DDMMYYYY en el archivo y pruebe de nuevo")
        
    return output_list


def get_station_date(Sample_Name):
    StationID = Sample_Name.split('_')[0]
    
    
    # HAY DOS TIPOS DE SAMPLE NAME, LOS QUE ACABAN EN SCAN Y LOS QUE NO
    # SCAN: TIENE EL FORMATO DE IDSTATION_DIA_MES_AÑO_SCAN
    # SV: TIENE EL FORMATO IDSTATION_DIAMESAÑO_FS_SV
    
    if len(Sample_Name.split('_')[1]) <= 2:
        day = Sample_Name.split('_')[1]
        month = Sample_Name.split('_')[2]
        year = Sample_Name.split('_')[3]
    else:
        day = Sample_Name.split('_')[1][0:2]
        month = Sample_Name.split('_')[1][2:4]
        year = '20'+Sample_Name.split('_')[1][4:6]
    dt = datetime(int(year), int(month), int(day))
    
    return StationID,dt.date()

In [8]:
# Proceso de insercción

excels_path_input = "../Datos Excel/LecturasIncluidas"
excel_number = 0
compound_tuples = 0
sample_tuples = 0

for excel_name in os.listdir(excels_path_input):
    
    # Obtenemos la ubicación del excel
    excel_path_input = os.path.join(excels_path_input, excel_name)
    excel_number += 1
    print(f"\nLoading Excel {excel_number} of {len(os.listdir(excels_path_input))}...")
    
    # Cargamos el primer excel
    workbook = openpyxl.load_workbook(excel_path_input)
    
    compound_tuples_aux = 0
    sample_tuples_aux = 0
    i = 0
    total_rows = sum(sheet.max_row - 4 for sheet in workbook.worksheets)
    
    # Creamos las columnas Date y Group
    for sheet in workbook.worksheets:
        # Si la hoja no es valida, pasamos a la siguiente
        if not sheet_is_valid(sheet):
            continue

        
        for row in sheet.iter_rows(min_row=5, max_row=sheet.max_row):
            i+=1
            print(f"\r\tRow {i}/{total_rows} - {((i/total_rows)*100):.2f}%", end="")  # Sobrescribe la línea con el nuevo progreso
            # row[0] = Component RT | row[1] = Library RT 	| row[2] = Compound Name	| row[3] = Match Factor	
            # row[4] = Formula	    | row[5] = CAS	        | row[6] = Sample Name
            
            # Antes de realizar la carga y la insercción comprobamos que la fila tiene datos
            if not row_is_valid(row):
                continue
            
            compound = Compounds(
                cas=row[5].value, 
                name=row[2].value, 
                formula=row[4].value, 
                group=None) # por el momento lo dejamos en None / NULL
            
            library_rt_value = row[1].value if row[1].value != '' else None
            StationID,dt = get_station_date(row[6].value)
            
            if StationID == '2571b':
                StationID = '2571'
            
            existing_station = session.query(Stations).filter(Stations.station_id == StationID).first()
            if not existing_station:
                continue
            
            sample = Samples(                
                station_id=StationID,
                compound_cas=row[5].value, 
                component_rt=row[0].value, 
                library_rt=library_rt_value,
                match_factor=row[3].value,
                sample_date=dt)
            
            
            # Verificar si el compuesto ya existe
            existing_compound = session.query(Compounds).filter_by(cas=row[5].value).first()
            if not existing_compound:
                session.add(compound)
                compound_tuples_aux+=1
            
            # Verificar si la muestra ya existe
            existing_sample = session.query(Samples).filter_by(
                station_id=StationID, 
                compound_cas=row[5].value,
                component_rt=row[0].value,
                library_rt=library_rt_value,
                match_factor=row[3].value,
                sample_date=dt
            ).first()
            
            
            if not existing_sample:
                session.add(sample)
                sample_tuples_aux+=1
    
    compound_tuples += compound_tuples_aux
    sample_tuples += sample_tuples_aux

# Guardar los cambios en la base de datos y liberar todas las conexiones
session.commit()
session.close()
engine.dispose()
         
print("\nAll tuples were successfully inserted into the database.")
print(f"{compound_tuples} tuples inserted into Compounds")
print(f"{sample_tuples} tuples inserted into Samples")


Loading Excel 1 of 19...
	Row 7010/7010 - 100.00%
Loading Excel 2 of 19...
	Row 6918/6918 - 100.00%
Loading Excel 3 of 19...
	Row 6842/6842 - 100.00%
Loading Excel 4 of 19...
	Row 459/474 - 96.84%
Loading Excel 5 of 19...
	Row 5849/5850 - 99.98%
Loading Excel 6 of 19...
	Row 4464/4464 - 100.00%
Loading Excel 7 of 19...
	Row 6809/6809 - 100.00%
Loading Excel 8 of 19...
	Row 6978/6978 - 100.00%
Loading Excel 9 of 19...
	Row 6523/6523 - 100.00%
Loading Excel 10 of 19...
	Row 5708/5708 - 100.00%
Loading Excel 11 of 19...
	Row 3859/3859 - 100.00%
Loading Excel 12 of 19...
	Row 5728/5728 - 100.00%
Loading Excel 13 of 19...
	Row 5694/5694 - 100.00%
Loading Excel 14 of 19...
	Row 2744/2744 - 100.00%
Loading Excel 15 of 19...
	Row 5542/5542 - 100.00%
Loading Excel 16 of 19...
	Row 4068/4068 - 100.00%
Loading Excel 17 of 19...
	Row 2425/2425 - 100.00%
Loading Excel 18 of 19...
	Row 2105/2105 - 100.00%
Loading Excel 19 of 19...
	Row 3774/3774 - 100.00%
All tuples were successfully inserted into 

In [9]:
# Consultamos las primeras 10 tuplas de la tabla 'samples' para confirmar que los datos se han insertado correctamente
engine = create_engine("sqlite:///../Database/Petrola.db")

Session = sessionmaker(bind=engine)
session = Session()

samples = session.query(Samples).filter(Samples.id < 11)

for sample in samples:
    print(f"{sample.id} - {sample.station_id} - {sample.compound_cas} - {sample.match_factor} - {sample.sample_date}")

session.close()
engine.dispose()

1 - 2554 - 630-02-4 - 97.3448758643285 - 2018-03-07
2 - 2554 - 3622-84-2 - 97.3426256374860 - 2018-03-07
3 - 2554 - 143-07-7 - 97.0340881226838 - 2018-03-07
4 - 2554 - 84-66-2 - 96.8800885288180 - 2018-03-07
5 - 2554 - 57-11-4 - 96.8264123682283 - 2018-03-07
6 - 2554 - 629-76-5 - 96.7604533453609 - 2018-03-07
7 - 2554 - 84-74-2 - 96.7198925332146 - 2018-03-07
8 - 2554 - 111-02-4 - 96.3098274851958 - 2018-03-07
9 - 2554 - 112-92-5 - 96.1331181349050 - 2018-03-07
10 - 2554 - 119-61-9 - 96.0722380246357 - 2018-03-07
